# 实验一：PASCAL VOC 数据集准备与 YOLO 配置

本章解决三个问题：

1. 为什么当前实验使用 PASCAL VOC，而不是继续使用完整 COCO2017。
2. VOC 原始标注是什么结构，为什么要转换为 YOLO txt。
3. 如何检查 `src/configs/yolo_ascend.yaml`，确认训练脚本能读取图片、标签和类别信息。

当前硬件是 `1*NPU 910B3 + 16 vCPU + 32GiB`，目标是先跑通完整训练流程，再做 AMP、Batch Size、DataLoader worker、Warmup 和 MSPROF 调优。VOC 的规模刚好适合这个目标。


## 为什么选择 PASCAL VOC

PASCAL VOC 是目标检测领域的经典数据集，常用于检测算法入门、训练流程验证和轻量调优。它包含 20 个类别，既有 `person`、`car` 这类常见目标，也有 `bottle`、`chair`、`dog` 等不同尺度的目标。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">选择理由</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">数据量适中</td>
      <td style="text-align: left;">VOC2007+VOC2012 下载约 2.8GB，保留 tar 后通常 8 到 10GB 左右，比完整 COCO2017 更稳妥</td>
    </tr>
    <tr>
      <td style="text-align: left;">类别清晰</td>
      <td style="text-align: left;">VOC 固定 20 类，配置和 label 检查更直观</td>
    </tr>
    <tr>
      <td style="text-align: left;">调优有效</td>
      <td style="text-align: left;">图片数量足够让 DataLoader、图片解码、NPU 计算和 AMP 差异变得可观察</td>
    </tr>
    <tr>
      <td style="text-align: left;">教学友好</td>
      <td style="text-align: left;">官方 XML 标注结构清楚，适合讲清楚“标注转换”这个新概念</td>
    </tr>
  </tbody>
</table>

COCO2017 更适合后续多卡或更大磁盘条件下做正式大规模实验。现在先用 VOC 跑通流程，后面扩展到 COCO 时，训练脚本、AMP、MSPROF 的思路不需要重学。


## 数据准备流程图

![PASCAL VOC 数据准备流程](images/voc_dataset_pipeline.svg)

图中的 `XML` 是一种结构化文本格式。VOC 的每张图片都有一个同名 XML 文件，里面记录图片尺寸、目标类别、目标框坐标等信息。训练脚本不直接读 XML，而是读取 YOLO txt，所以需要 `prepare_voc_yolo.py` 做格式转换。


## VOC 原始目录是什么

下载并解压后，原始 VOC 通常长这样：

```text
/mnt/workspace/datasets/voc/
└── VOCdevkit/
    ├── VOC2007/
    │   ├── JPEGImages/        # 图片文件
    │   ├── Annotations/       # XML 标注文件
    │   └── ImageSets/Main/    # trainval、test 等划分列表
    └── VOC2012/
        ├── JPEGImages/
        ├── Annotations/
        └── ImageSets/Main/
```

第一次出现的几个词先解释清楚：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">名称</th>
      <th style="text-align: left;">含义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>JPEGImages</code></td>
      <td style="text-align: left;">存放 <code>.jpg</code> 图片</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>Annotations</code></td>
      <td style="text-align: left;">存放 <code>.xml</code> 标注，每张图片一个同名 XML</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>ImageSets/Main</code></td>
      <td style="text-align: left;">存放数据划分列表，例如 <code>trainval.txt</code>、<code>test.txt</code></td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>trainval</code></td>
      <td style="text-align: left;">训练集和验证集合并后的官方划分，本实验用作训练数据</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>test</code></td>
      <td style="text-align: left;">VOC2007 的测试集，本实验用作验证数据</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>split</code></td>
      <td style="text-align: left;">训练脚本使用的图片路径列表，每一行是一张图片</td>
    </tr>
  </tbody>
</table>


## 下载 VOC

在服务器上进入实验目录后运行：

```bash
bash src/scripts/download_voc.sh
```

这个脚本默认把数据放到 `/mnt/workspace/datasets/voc`。如果要换位置，可以先设置 `VOC_ROOT`：

```bash
VOC_ROOT=/mnt/workspace/datasets/voc bash src/scripts/download_voc.sh
```

如果下载地址访问慢或失败，可以手动上传这三个 tar 文件到 `/mnt/workspace/datasets/voc`：

```text
VOCtrainval_06-Nov-2007.tar
VOCtest_06-Nov-2007.tar
VOCtrainval_11-May-2012.tar
```

上传后再运行转换脚本即可。


## XML 转 YOLO txt

YOLO txt 的每一行表示一个目标：

```text
class_id x_center y_center width height
```

这些坐标都是归一化坐标，也就是除以图片宽高后的比例值，通常在 0 到 1 之间。归一化后，同一个模型就可以处理不同原始尺寸的图片。

运行转换：

```bash
python src/scripts/prepare_voc_yolo.py --root /mnt/workspace/datasets/voc
```

常用参数说明：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">参数</th>
      <th style="text-align: left;">作用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>--root</code></td>
      <td style="text-align: left;">VOC 数据根目录</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>--train-limit</code></td>
      <td style="text-align: left;">限制训练图片数量，<code>0</code> 表示不限制</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>--val-limit</code></td>
      <td style="text-align: left;">限制验证图片数量，<code>0</code> 表示不限制</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>--copy-images</code></td>
      <td style="text-align: left;">复制图片而不是创建软链接；软链接能节省磁盘空间</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>--include-difficult</code></td>
      <td style="text-align: left;">是否保留 VOC 中标为 difficult 的目标</td>
    </tr>
  </tbody>
</table>

`软链接` 可以理解成“指向原图片的快捷入口”。它不会复制图片内容，因此很省空间；如果系统不允许创建软链接，脚本会自动退回到复制图片。


In [ ]:
# ====== 1. 读取实验配置 ======
from pathlib import Path
import yaml

config_path = Path('src/configs/yolo_ascend.yaml')
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))

print('项目名称:', cfg['project']['name'])
print('数据根目录:', cfg['data']['root'])
print('训练 split:', cfg['data']['train_list'])
print('验证 split:', cfg['data']['val_list'])
print('类别数量:', cfg['model']['num_classes'])
print('输入尺寸:', cfg['data']['image_size'])
print('Batch Size / rank:', cfg['train']['batch_size'])
print('AMP:', cfg['train']['amp'])


In [ ]:
# ====== 2. 检查 VOC 图片、标签和 split 是否就绪 ======
from pathlib import Path

root = Path(cfg['data']['root'])
checks = {
    'VOC2007 原始目录': root / 'VOCdevkit' / 'VOC2007',
    'VOC2012 原始目录': root / 'VOCdevkit' / 'VOC2012',
    '训练图片目录': root / 'images',
    '训练标签目录': root / 'labels',
    '训练 split': root / cfg['data']['train_list'],
    '验证 split': root / cfg['data']['val_list'],
    '类别文件': root / 'voc.names',
}

for name, path in checks.items():
    print(f'{name:14s}: {path} -> {path.exists()}')

if (root / cfg['data']['train_list']).exists():
    train_lines = (root / cfg['data']['train_list']).read_text(encoding='utf-8').splitlines()
    print('训练样本数:', len([line for line in train_lines if line.strip()]))

if (root / cfg['data']['val_list']).exists():
    val_lines = (root / cfg['data']['val_list']).read_text(encoding='utf-8').splitlines()
    print('验证样本数:', len([line for line in val_lines if line.strip()]))


In [ ]:
# ====== 3. 查看一条 YOLO label ======
from pathlib import Path

train_split = root / cfg['data']['train_list']
if train_split.exists():
    first_image = next(line.strip() for line in train_split.read_text(encoding='utf-8').splitlines() if line.strip())
    image_path = root / first_image
    label_path = root / 'labels' / Path(first_image).relative_to('images')
    label_path = label_path.with_suffix('.txt')
    print('image:', image_path)
    print('label:', label_path)
    if label_path.exists():
        print(label_path.read_text(encoding='utf-8').splitlines()[:5])
    else:
        print('label 文件不存在，请重新运行 prepare_voc_yolo.py')
else:
    print('train split 不存在，请先运行 bash src/scripts/download_voc.sh 或 prepare_voc_yolo.py')


## YOLO 配置重点

`src/configs/yolo_ascend.yaml` 是训练脚本的总开关。先理解下面几个字段：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">字段</th>
      <th style="text-align: left;">当前值</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>data.root</code></td>
      <td style="text-align: left;"><code>/mnt/workspace/datasets/voc</code></td>
      <td style="text-align: left;">数据集根目录</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>data.train_list</code></td>
      <td style="text-align: left;"><code>splits/train.txt</code></td>
      <td style="text-align: left;">训练图片路径列表</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>data.val_list</code></td>
      <td style="text-align: left;"><code>splits/val.txt</code></td>
      <td style="text-align: left;">验证图片路径列表</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>model.num_classes</code></td>
      <td style="text-align: left;"><code>20</code></td>
      <td style="text-align: left;">VOC 类别数</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>train.batch_size</code></td>
      <td style="text-align: left;"><code>16</code></td>
      <td style="text-align: left;">单次送入 NPU 的图片数量</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>train.workers</code></td>
      <td style="text-align: left;"><code>8</code></td>
      <td style="text-align: left;">DataLoader 读取图片的子进程数量</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>train.amp</code></td>
      <td style="text-align: left;"><code>true</code></td>
      <td style="text-align: left;">开启混合精度训练</td>
    </tr>
  </tbody>
</table>

`DataLoader` 是 PyTorch 的数据读取器。`workers` 越大，CPU 侧并行读图和解码能力通常越强，但也会消耗更多 CPU 和内存。你当前是 16 vCPU，先从 `workers=8` 开始比较稳妥。


In [ ]:
# ====== 4. 轻量检查 DataLoader 是否能构建 ======
import sys
from pathlib import Path

sys.path.insert(0, str(Path('src/scripts').resolve()))
try:
    from train_yolo_ddp_amp import load_config, make_dataset

    train_dataset = make_dataset(load_config('src/configs/yolo_ascend.yaml'), 'train')
    val_dataset = make_dataset(load_config('src/configs/yolo_ascend.yaml'), 'val')
    print('train_dataset:', len(train_dataset))
    print('val_dataset  :', len(val_dataset))
    image, target = train_dataset[0]
    print('one image tensor:', tuple(image.shape), image.dtype)
    print('one target shape:', tuple(target.shape))
except Exception as exc:
    print('DataLoader 构建失败:', exc)
    print('优先检查 VOC 是否已下载、prepare_voc_yolo.py 是否已运行、config 路径是否一致。')


## 本章小结

本章完成了数据集选择、VOC 原始结构说明、XML 到 YOLO txt 的转换说明，以及训练配置检查。下一章开始解释单卡 NPU 的启动方式，并说明为什么当前不直接运行多卡 DDP。


## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) PASCAL VOC 原始目标检测标注主要采用哪种格式？
   - A. JSON
   - B. XML
   - C. CSV
   - D. NPZ

2. (单选题) YOLO txt 每一行通常表示什么？
   - A. 一张图片
   - B. 一个目标框
   - C. 一个 batch
   - D. 一个模型权重

3. (单选题) 把 VOC XML 转为 YOLO txt 的核心原因是？
   - A. 减小图片尺寸
   - B. 让训练脚本读取统一的 YOLO 标注格式
   - C. 提高 Git 上传速度
   - D. 替代模型结构

4. (单选题) `prepare_voc_yolo.py` 的输出不包括哪一项？
   - A. images/train2007、images/train2012、images/val2007
   - B. labels/.../*.txt
   - C. splits/train.txt 与 splits/val.txt
   - D. CANN .om 模型

5. (多选题) 关于 VOC2007 与 VOC2012，下列说法正确的是？
   - A. 它们是 PASCAL VOC 系列中的两个年份版本
   - B. 训练时可以组合 VOC2007+VOC2012 的 trainval 数据
   - C. VOC2007 test 常用于评估
   - D. VOC2012 天然包含 VOC2007 的所有图片

6. (多选题) YOLO txt 中的坐标为什么要归一化到 0 到 1？
   - A. 便于适配不同图片尺寸
   - B. 避免模型只适用于固定原图宽高
   - C. 让标注和 resize 后的输入更容易统一处理
   - D. 为了让类别数变少

7. (多选题) 数据准备后，应重点检查哪些文件或目录？
   - A. images/ 是否有图片
   - B. labels/ 是否有同名 txt 标注
   - C. splits/train.txt 和 splits/val.txt 是否存在
   - D. 模型 checkpoint 是否一定已经生成

8. (判断题) XML 转 YOLO txt 属于训练前的数据预处理步骤。

9. (判断题) YOLO txt 中 width 和 height 表示像素宽高，通常不需要除以图片宽高。

10. (填空题) VOC XML 中的 `xmin,ymin,xmax,ymax` 会被转换为 YOLO txt 中的 `____, ____, ____, ____`。

11. (填空题) 类别名称列表通常写入 `____`，训练配置中的 `num_classes` 应与类别数量保持一致。

12. (简答题) 为什么训练配置中的数据路径必须和实际生成目录保持一致？

13. (简答题) 如果转换后某些图片没有生成对应 txt，可能有哪些原因？

14. (简答题) 为什么数据准备阶段不建议直接修改原始 VOCdevkit 目录？

15. (代码设计题) 写一段 Python 代码，检查 `splits/train.txt` 中前 5 个样本是否都有图片和标签文件。

> 参考答案见 answer/03.02_dataset_preparation_and_yolo_config_answer.ipynb。
